# Clase 4: Proporciones y relaciones — Pie, Box, Scatter y Bubble plots

## ¿Por qué te importa esto?

No todas las preguntas de datos son "¿cuánto?" o "¿cómo cambia con el tiempo?". A veces la pregunta real es:

- ¿**Qué proporción** del total le corresponde a cada grupo? → **pie chart**
- ¿Cómo se **distribuyen** mis datos y hay algún valor raro entre ellos? → **box plot**
- ¿Existe una **relación** entre dos variables distintas? → **scatter plot**
- ¿Esa relación cambia si además tenemos en cuenta una **tercera variable**? → **bubble plot**

Estas cuatro herramientas cierran el círculo del análisis exploratorio: ya sabemos contar tendencias (líneas), comparar categorías (barras) y ver distribuciones (histogramas). Ahora aprendemos a ver **reparto, dispersión y relación**.

Seguimos trabajando con `df_can` (inmigración a Canadá, 1980-2013), la lista `years` y el índice por país.

## Preparación del entorno y los datos

In [ ]:
# Import primary modules
import numpy as np   # útil para cálculo científico (lo usaremos para np.polyfit)
import pandas as pd  # librería principal de estructuras de datos

# Importing Matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('ggplot')  # opcional: estilo visual tipo ggplot

print('Matplotlib version: ', mpl.__version__)

Descargamos el dataset ya limpio de inmigración a Canadá (mismo archivo usado en clases anteriores).

In [ ]:
df_can = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/Data%20Files/Canada.csv')

print('Datos cargados en un DataFrame de pandas!')
df_can.head()

In [ ]:
# Fijamos 'Country' como índice para poder buscar países con .loc
df_can.set_index('Country', inplace=True)

# Lista de años 1980-2013 como strings (nombres de columnas)
years = list(map(str, range(1980, 2014)))

print('Dimensiones:', df_can.shape)
df_can.head()

## Pie charts: repartir la pizza

Un pie chart es literalmente una **pizza** que repartes entre varios comensales. Cada porción es proporcional al apetito (o al peso) de cada comensal dentro del total. Es la herramienta correcta cuando quieres mostrar **de qué está hecho el 100%**, no cuando quieres comparar magnitudes absolutas grandes (para eso, usa barras).

Vamos a ver qué proporción de la inmigración total a Canadá (1980-2013) vino de cada continente. Primero agrupamos con `groupby` (dividir → aplicar → combinar: divide el dataframe en grupos según `Continent`, aplica `.sum()` a cada grupo, y combina los resultados).

In [ ]:
# agrupar países por continente y sumar sus valores
df_continents = df_can.groupby('Continent').sum(numeric_only=True)
df_continents.head()

In [ ]:
df_continents['Total'].plot(kind='pie',
                             figsize=(10, 6),
                             autopct='%1.1f%%',   # muestra el porcentaje dentro de cada porción
                             startangle=90,        # empieza a dibujar desde 90°
                             shadow=True)           # sombra para dar sensación de 3D

plt.title('Inmigración a Canadá por continente (1980-2013)')
plt.axis('equal')  # fuerza a que el círculo se vea como círculo, no como óvalo
plt.legend(labels=df_continents.index, loc='upper left')

plt.show()

Si los porcentajes se amontonan o se solapan con las etiquetas, los separamos con `pctdistance` y usamos `explode` para "sacar" una porción hacia afuera y llamar la atención sobre ella (como sacar una porción de pizza del centro del plato).

In [ ]:
explode_list = [0.1, 0, 0, 0, 0.1, 0.1]  # cuánto se separa cada porción del centro (0 = no se mueve)

df_continents['Total'].plot(kind='pie',
                             figsize=(10, 6),
                             autopct='%1.1f%%',
                             startangle=90,
                             shadow=True,
                             labels=None,          # quitamos las etiquetas encima del pie
                             pctdistance=1.12,     # empuja el porcentaje fuera de la porción
                             colors=['gold', 'yellowgreen', 'lightcoral', 'lightskyblue', 'lightgreen', 'pink'],
                             explode=explode_list)

plt.title('Inmigración a Canadá por continente (1980-2013)', y=1.12, fontsize=15)
plt.axis('equal')
plt.legend(labels=df_continents.index, loc='upper left', fontsize=7)

plt.show()

## Box plots: la mediana y sus cuartos

Imagina las notas de un examen de 30 estudiantes ordenadas de menor a mayor. La nota que queda justo en el medio es la **mediana**. Si divides esa lista ordenada en cuatro partes iguales, los puntos de corte son los **cuartiles**: el primer cuartil (Q1) deja el 25% de las notas por debajo, el tercer cuartil (Q3) deja el 75% por debajo. Un box plot dibuja exactamente eso: una caja que va de Q1 a Q3 (con la mediana marcada dentro), y unos "bigotes" que se extienden hasta el mínimo y el máximo razonables.

Vamos a ver el número de inmigrantes que llegó desde Japón, 1980-2013.

In [ ]:
# extraerlo como dataframe (con doble corchete) para poder usar describe() después
df_japan = df_can.loc[['Japan'], years].transpose()

df_japan.plot(kind='box', figsize=(8, 6))

plt.title('Box plot de inmigrantes japoneses (1980-2013)')
plt.ylabel('Número de inmigrantes')

plt.show()

# ver los números exactos detrás de la caja
df_japan.describe()

De un vistazo se lee: el mínimo ronda 200, el máximo ronda 1300, la mediana ronda 900; el 25% de los años tuvo 500 inmigrantes o menos, el 75% tuvo 1100 o menos.

### Outliers: los valores que se salen de la norma

Un box plot no solo resume: también **detecta valores atípicos** automáticamente. La regla es matemática, no arbitraria. Primero se calcula el rango intercuartílico:

```
IQR = Q3 - Q1
```

Y un valor se marca como **outlier** si:

- es mayor que `Q3 + 1.5 × IQR`, o
- es menor que `Q1 - 1.5 × IQR`

Por ejemplo, si comparamos las 15 principales fuentes de inmigración agrupadas por décadas y en la década de 2000 tenemos Q1 = 36,101.5 y Q3 = 105,505.5, el IQR es 69,404. Cualquier país con más de `105,505.5 + 1.5 × 69,404 = 209,611.5` inmigrantes en esa década aparece en el gráfico como un punto suelto por encima del bigote superior: un outlier real, no un error de cálculo. En este dataset, **China e India** superan ese umbral.

Comparemos ambos países en el mismo box plot, dibujado horizontal con `vert=False`.

In [ ]:
df_CI = df_can.loc[['China', 'India'], years].transpose()

df_CI.plot(kind='box', figsize=(10, 7), color='blue', vert=False)

plt.title('Box plots de inmigrantes de China e India (1980-2013)')
plt.xlabel('Número de inmigrantes')

plt.show()

df_CI.describe()

### Subplots: varios gráficos, una sola figura

Cuando queremos comparar dos vistas del mismo dato lado a lado (el box plot y la línea temporal de China e India), usamos **subplots**. Trabajamos con la capa de artista (`fig`, `ax`) en vez de `plt` directo: el truco es pasar `ax=ax0` o `ax=ax1` al método `.plot()` de cada dataframe, para decirle a *pandas* en qué subgráfico dibujar.

In [ ]:
fig = plt.figure(figsize=(20, 6))

ax0 = fig.add_subplot(1, 2, 1)  # 1 fila, 2 columnas, primer gráfico
ax1 = fig.add_subplot(1, 2, 2)  # 1 fila, 2 columnas, segundo gráfico

df_CI.plot(kind='box', color='blue', vert=False, ax=ax0)
ax0.set_title('Box plots de China e India (1980-2013)')
ax0.set_xlabel('Número de inmigrantes')

df_CI.plot(kind='line', ax=ax1)
ax1.set_title('Evolución anual de China e India (1980-2013)')
ax1.set_ylabel('Número de inmigrantes')
ax1.set_xlabel('Años')

plt.show()

## Scatter plots: la relación entre dos variables

Un scatter plot responde a una pregunta distinta: ¿existe relación entre dos variables numéricas? Piensa en el ejemplo clásico de altura y peso de un grupo de personas: si dibujas un punto por persona (altura en x, peso en y), verás si tienden a subir juntas. A diferencia de un gráfico de líneas, los puntos de un scatter **no se conectan entre sí**: cada punto es una observación independiente, y el patrón (si existe) emerge de la nube de puntos, no de una línea que los una.

Veamos la relación entre el año y el total de inmigrantes a Canadá.

In [ ]:
# sumar todos los países para cada año
df_tot = pd.DataFrame(df_can[years].sum(axis=0))
df_tot.index = map(int, df_tot.index)   # convertir el año a entero, lo necesitaremos para la regresión
df_tot.reset_index(inplace=True)
df_tot.columns = ['year', 'total']

df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')

plt.title('Inmigración total a Canadá (1980-2013)')
plt.xlabel('Año')
plt.ylabel('Número de inmigrantes')

plt.show()

Se ve una tendencia ascendente clara. Un scatter plot nos deja **ver** la relación, pero si queremos cuantificarla y usarla para predecir, necesitamos una regresión.

### Regresión lineal manual con `np.polyfit`

`np.polyfit(x, y, deg=1)` calcula, por mínimos cuadrados, los coeficientes de la recta que mejor se ajusta a los puntos (`deg=1` significa grado 1: una línea recta; `deg=2` sería una parábola, y así sucesivamente).

In [ ]:
x = df_tot['year']
y = df_tot['total']

fit = np.polyfit(x, y, deg=1)
fit  # array([pendiente, intercepto])

El resultado son los coeficientes de `y = pendiente * x + intercepto`. Dibujamos la recta encima del scatter con `plt.plot`.

In [ ]:
df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')

plt.title('Inmigración total a Canadá (1980-2013)')
plt.xlabel('Año')
plt.ylabel('Número de inmigrantes')

# la recta de regresión
plt.plot(x, fit[0] * x + fit[1], color='red')
plt.annotate('y={0:.0f} x + {1:.0f}'.format(fit[0], fit[1]), xy=(2000, 150000))

plt.show()

# predicción simple para el año 2015
prediccion_2015 = fit[0] * 2015 + fit[1]
print(f'Predicción de inmigrantes totales para 2015: {prediccion_2015:,.0f}')

Con esta ecuación podemos hacer una predicción simple: sustituyendo `year=2015` obtenemos una estimación de inmigrantes para ese año, que podríamos comparar contra el dato real y ver qué tan bien (o mal) se ajusta una línea recta a la realidad.

Hacer esto a mano con `np.polyfit` importa por una razón concreta: nos obliga a entender qué es una regresión —una recta que minimiza el error total— antes de usar, más adelante, librerías como Seaborn que la calculan y dibujan con una sola llamada (`sns.regplot`, por ejemplo).

## Bubble plots: scatter plots con una tercera historia

Un bubble plot es un scatter plot al que le añadimos una dimensión más: el **tamaño** de cada punto también significa algo. Si el scatter normal dice "dónde está cada dato en x e y", el bubble plot añade "y además, qué tan grande es ese dato en una tercera variable z". Es como un scatter de altura contra peso donde, además, el tamaño de cada punto representa la edad de la persona: de un vistazo vemos tres variables a la vez.

Comparemos la inmigración desde **Brasil y Argentina**, usando el tamaño de la burbuja para representar la magnitud de cada año.

Primero, preparamos los datos transpuestos con el año como columna.

In [ ]:
df_can_t = df_can[years].transpose()
df_can_t.index = map(int, df_can_t.index)
df_can_t.index.name = 'Year'
df_can_t.reset_index(inplace=True)

df_can_t.head()

### Normalizar el tamaño

Los valores brutos de inmigrantes no sirven directamente como tamaño de punto: unos son demasiado pequeños para verse y otros demasiado grandes para caber en pantalla. Por eso normalizamos cada serie al rango [0, 1] con *feature scaling*: el valor mínimo se convierte en 0, el máximo en 1, y todo lo demás se reparte proporcionalmente entre medio.

In [ ]:
# normalizar Brasil
norm_brazil = (df_can_t['Brazil'] - df_can_t['Brazil'].min()) / (df_can_t['Brazil'].max() - df_can_t['Brazil'].min())

# normalizar Argentina
norm_argentina = (df_can_t['Argentina'] - df_can_t['Argentina'].min()) / (df_can_t['Argentina'].max() - df_can_t['Argentina'].min())

norm_brazil.head()

Como el resultado normalizado vive entre 0 y 1, si lo usamos tal cual como tamaño de punto (`s`) casi no se vería en el gráfico. Por eso lo escalamos: lo multiplicamos por un factor grande y le sumamos un mínimo para que hasta el valor 0 se vea como un punto pequeño, no invisible.

In [ ]:
# Brasil
ax0 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='Brazil',
                     figsize=(14, 8),
                     alpha=0.5,
                     color='green',
                     s=norm_brazil * 2000 + 10,   # tamaño = normalizado * 2000 + 10 (para que no desaparezca)
                     xlim=(1975, 2015))

# Argentina, en los mismos ejes que Brasil (ax=ax0)
ax1 = df_can_t.plot(kind='scatter',
                     x='Year',
                     y='Argentina',
                     alpha=0.5,
                     color='blue',
                     s=norm_argentina * 2000 + 10,
                     ax=ax0)

ax0.set_ylabel('Número de inmigrantes')
ax0.set_title('Inmigración desde Brasil y Argentina (1980-2013)')
ax0.legend(['Brasil', 'Argentina'], loc='upper left', fontsize='x-large')

plt.show()

El resultado muestra burbujas más grandes durante la crisis argentina de 1998-2002 y durante la crisis del real brasileño en 1999: el **tamaño de la burbuja cuenta, literalmente, la magnitud de cada crisis migratoria** año a año, algo que un scatter plano no podría transmitir.

## Puntos clave

- El **pie chart** muestra **proporciones de un total**; usa `explode` para destacar una porción y `autopct` para mostrar el porcentaje directamente sobre el gráfico.
- El **box plot** resume una distribución en cinco números (mínimo, Q1, mediana, Q3, máximo) y **detecta outliers automáticamente** con la regla de 1.5 × IQR.
- El **scatter plot** revela **relaciones entre dos variables numéricas**; los puntos no se conectan entre sí.
- `np.polyfit(x, y, deg=1)` calcula a mano la recta de regresión; entenderlo ahora prepara para usar herramientas de regresión automáticas más adelante.
- El **bubble plot** es un scatter plot con una tercera variable codificada en el **tamaño** del punto; siempre normaliza y reescala esa variable antes de pasarla al parámetro `s`.
- Los **subplots** (`fig.add_subplot()`) permiten comparar varias vistas del mismo dato en una sola figura.

## Siguiente paso

En la práctica aplicarás estas cuatro herramientas sobre el mismo dataset `df_can`, con nuevos recortes de datos (año 2013, China vs. India).